In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import duckdb as ddb
import numpy as np

ActualForecast = pd.read_csv('ActualForecastData.csv', parse_dates=['date_he'])

print(ActualForecast.info())
print(ActualForecast.head())
print(ActualForecast.tail())

#### Average demand by hour throughout the week

In [ ]:
day_map = {
    0: 'Sunday',
    1: 'Monday',
    2: 'Tuesday',
    3: 'Wednesday',
    4: 'Thursday',
    5: 'Friday',
    6: 'Saturday'
}
hourly_demand = ddb.sql('''
    select
        case when hour(date_he) = 0 then 24 else hour(date_he) end as hour,
        dayofweek(date_he) as day_of_week,
        avg(actual_ail) as avg_ail
    from ActualForecast
    group by hour, day_of_week
    order by hour, day_of_week
''').df()
hourly_demand = hourly_demand.pivot(index='hour', columns='day_of_week', values='avg_ail')
hourly_demand = hourly_demand.rename(columns=day_map)
hourly_demand = hourly_demand[['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']]

#################### PLOT ##############################
hourly_demand.plot(figsize=(12, 6), marker='o')
plt.xlim(1, 24)
plt.xticks(range(1, 25))
plt.title('Average Hourly Demand by Day of the Week')
plt.ylabel('Average Actual AIL MW')
plt.xlabel('Hour Ending')
plt.grid(True)
plt.legend(loc = 'upper left')

As a utility, electricity demand follows predictable patterns that smooth over time. Hourly demand shows a persistent structure: a morning spike from hours 6 to 8, a steady climb to peak demand from hours 16 to 18, then a gradual decline into a trough in the early morning hours.

Weekday volumes are closely aligned, while weekends show a significant drop in demand — though the underlying shape remains consistent.

Residential and commercial consumption drive these observed patterns. However, Alberta's heavy reliance on oil and gas introduces a substantial baseload component, as oil sands extraction requires continuous electricity regardless of time of day.

#### Average hourly demand throughout the year

In [ ]:
avg_hourly_demand = ddb.sql('''
    select
        year(date_he) as year,
        dayofyear(date_he) as doy,
        avg(actual_ail) as avg_ail
    from ActualForecast
    group by doy, year
    order by doy, year
    ''').df()

avg_hourly_demand = avg_hourly_demand.pivot(index='doy', columns='year', values='avg_ail')

fig, ax = plt.subplots(figsize=(12, 6))
avg_hourly_demand.plot(ax=ax, alpha=1, color=['blue', 'purple', 'red', 'maroon'], linewidth=0.9)
ax.set_xlabel('Day')
ax.set_ylabel('MW')
ax.set_title('Average Hourly Demand Throughout the Year')
ax.set_xlim(1, 366)
ax.set_xticks(range(1, 367, 15))
ax.legend(title=None, loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Monthly demand patterns are consistent across the years. Weather conditions corresponds to heating and cooling needs as demand peaks during winter, Dec to Feb, and summer, July and August. Demand declines during milder seasons, usually reaching its lowest in May or September.

#### Hourly price-demand trend

In [ ]:
hourly_price_demand = ddb.sql('''
    select
        case when hour(date_he) = 0 then 24 else hour(date_he) end as hour,
        year(date_he) as year,
        avg(actual_ail) as avg_ail,
        avg(actual_price) as avg_price
    from ActualForecast
    group by year, hour
    order by year, hour
    ''').df()

price_pivot = hourly_price_demand.pivot(index='hour', columns='year', values='avg_price')

colors = ['blue', 'purple', 'red', 'maroon']

fig, ax = plt.subplots(figsize=(12, 6))
price_pivot.plot(ax=ax, alpha=1, color=colors, marker='o', linewidth=0.9)
ax.set_xlabel('Hour Ending')
ax.set_ylabel('$/MW')
ax.set_title('Average Price by Hour')
ax.set_xlim(1, 24)
ax.set_xticks(range(1, 25))
# Relabel the most recent year as partial
labels = [f'{int(y)} (YTD, partial)' if int(y) == 2026 else str(int(y))
          for y in price_pivot.columns]
ax.legend(labels, title=None, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Hourly prices dropped significantly from 2023 to 2024-2026. With Alberta's coal phase out completion in 2024, natural gas fired generators now predominantly set the marginal price making natural gas prices a key driver of electricity cost. This decline in the pool prices is explained by the drop of natural gas prices from 2023 as well as the addition of new gas fired and renewable generators capacity to the grid. https://discoverairdrie.com/articles/albertas-wholesale-power-price-dropped-53-in-2024-as-new-supply-surged

Prices spike during the morning ramp from 6 to 8 AM and again at the evening peak at 6 PM. Throughout the day, prices stabilize along a downward curve. Notably, prices remain elevated from 6 to 9 PM, despite demand declining during those hours.

#### Weekly price-demand relationship

In [ ]:
weekly_price_demand = ddb.sql('''
    select
        week(date_he) as week,
        year(date_he) as year,
        avg(actual_ail) as avg_ail,
        avg(actual_price) as avg_price
    from ActualForecast
    group by year, week
    order by year, week
    ''').df()

years = sorted(weekly_price_demand['year'].unique())

for year in years:
    grp = weekly_price_demand[weekly_price_demand['year'] == year]
    x, y = grp['avg_ail'].values, grp['avg_price'].values

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.scatter(x, y, color='steelblue', alpha=0.8)

    # Line of best fit
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, p(xs), 'r--', linewidth=2)

    # R² for the writeup
    r2 = np.corrcoef(x, y)[0, 1] ** 2
    ax.text(0.05, 0.95, f'slope = {z[0]:.4f}\n$R^2$ = {r2:.2f}',
            transform=ax.transAxes, va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    ax.set_xlabel('Actual AIL (MW)')
    ax.set_ylabel('Price ($/MW)')
    ax.set_title(f'Average Weekly Demand vs Price {int(year)}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

The line of best fit on each graph trends toward a flat slope, indicating a weak linear relationship between price and demand. This is expected given Alberta's merit order market structure, where generator bids are sorted from least to most expensive, dispatched sequentially until demand is met, and the marginal generator sets the price. As a result, prices depend on the available supply stack rather than demand alone. This also explains why some weeks in the shoulder months can see elevated prices when a generator goes offline for maintenance, tightening the supply stack.

#### Price-renewable generation relationship

In [ ]:
power_gen = pd.read_csv('PowerGeneration.csv')


From this observation, renewable generation has steadily increased over the study period, with wind generation accounting for the majority of growth as Alberta transitions towards cleaner energy sources. The relationship between renewable energy and prices is inversely related judging by the negative correlation -0.68. Since renewable generation has near zero marginal cost, they are the first to dispatch in the merit order when available displacing higher cost natural gas generators resulting in a lower pool price. This supports the earlier statement that higher demand does not always result in higher prices when renewable generators are on the grid.

need to fix this!